# <font color="#418FDE" size="6.5" uppercase>**Pipelines bauen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Wenden Skalierer, Imputer und Encoder passend auf numerische und kategoriale Spalten an. 
- Verbinden Spaltentransformationen mit ColumnTransformer und Pipeline. 
- Untersuchen Pipeline-Schritte, verschachtelte Parameter und Merkmalsnamen. 


## **1. Vorverarbeitung anwenden**

### **1.1. Skalierer gezielt einsetzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_01_01.jpg?v=1784796128" width="250">



>* Skalierung macht numerische Merkmale vergleichbar
>* Große Werte dominieren Modelle sonst unnötig

>* Skalierer nach Verteilung und Modell wählen
>* Ausreißer, Einheiten und Grenzen beachten

>* Skalierer nur mit Trainingsdaten anpassen
>* Datenleckage vermeiden, Bewertungen glaubwürdig halten



In [ ]:
#@title Python-Code - Skalierer gezielt einsetzen

# Dieses Beispiel zeigt gezielte Skalierung numerischer Spalten.
# Unterschiedliche Größenordnungen werden vergleichbar gemacht.
# Die Ausgabe vergleicht Rohdaten und skalierte Werte.

import pandas as pd
from sklearn import __version__ as sklearn_version
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler

# Kleine Immobiliendaten zeigen verschiedene numerische Größenordnungen.
houses = pd.DataFrame(
    {
        "living_area_m2": [45, 80, 120, 160, 240],
        "year_built": [1955, 1980, 2005, 2015, 2022],
        "price_eur": [180000, 320000, 520000, 760000, 1500000],
    }
)

# Die Formprüfung macht die Beispielannahme sichtbar.
if houses.shape != (5, 3):
    raise ValueError("Die Beispieldaten sollten fünf Zeilen und drei Spalten haben.")

# Drei Skalierer passen zu unterschiedlichen Datensituationen.
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

# Jeder Skalierer lernt Kennwerte nur aus diesen Trainingsdaten.
scaled_tables = {}
for scaler_name, scaler in scalers.items():
    scaled_values = scaler.fit_transform(houses)
    scaled_tables[scaler_name] = pd.DataFrame(
        scaled_values,
        columns=houses.columns,
    ).round(2)

print(f"scikit-learn-Version: {sklearn_version}")
print("Rohdaten, erste drei Zeilen:")
print(houses.head(3).to_string(index=False))
print("StandardScaler, erste drei Zeilen:")
print(scaled_tables["StandardScaler"].head(3).to_string(index=False))
print("MinMaxScaler bringt jede Spalte in den Bereich 0 bis 1.")
print(scaled_tables["MinMaxScaler"].head(3).to_string(index=False))



### **1.2. Fehlende Werte ersetzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_01_02.jpg?v=1784796126" width="250">



>* Fehlende Werte entstehen aus vielen Gründen
>* Imputation beeinflusst Modelllernen und Fairness

>* Numerische Lücken mit Mittelwert oder Median füllen
>* Kategorien fachlich sinnvoll ersetzen oder markieren

>* Imputation nur mit Trainingsdaten lernen
>* Imputer sichern konsistente, reproduzierbare Vorverarbeitung



In [ ]:
#@title Python-Code - Fehlende Werte ersetzen

# Dieses Beispiel ersetzt fehlende Werte in Tabellenspalten.
# Numerische und kategoriale Spalten brauchen unterschiedliche Strategien.
# Die Ausgabe zeigt gelernte Ersatzwerte und transformierte Daten.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import sklearn

# Wir bauen einen kleinen Datensatz mit absichtlichen Lücken.
data = pd.DataFrame(
    {
        "age": [22, 35, np.nan, 41, 29],
        "income": [32000, np.nan, 45000, 52000, 39000],
        "city": ["Berlin", "Hamburg", np.nan, "Berlin", "Köln"],
    }
)

# Diese Spalten werden passend nach Datentyp verarbeitet.
numeric_columns = ["age", "income"]
categorical_columns = ["city"]

# Numerische Lücken ersetzen wir hier robust mit dem Median.
numeric_transformer = SimpleImputer(strategy="median")

# Kategoriale Lücken werden als eigene Kategorie sichtbar gemacht.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unbekannt")),
        ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
    ]
)

# ColumnTransformer wendet jede Strategie nur auf passende Spalten an.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns),
    ]
)

# Fit lernt Ersatzwerte nur aus den vorhandenen Trainingsdaten.
transformed_array = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()

# Eine kleine Prüfung macht die Demonstration nachvollziehbar.
if transformed_array.shape[0] != len(data):
    raise ValueError("Die Zeilenzahl passt nach der Transformation nicht.")

# Für Anfänger ist eine kleine Tabelle am übersichtlichsten.
transformed_data = pd.DataFrame(
    transformed_array,
    columns=feature_names,
)

# Wir zeigen nur wenige, gezielt ausgewählte Ergebnisse.
median_values = preprocessor.named_transformers_["num"].statistics_
city_imputer = preprocessor.named_transformers_["cat"].named_steps["imputer"]
city_fill_value = city_imputer.statistics_[0]

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Gelernte Mediane: age={median_values[0]:.1f}, income={median_values[1]:.0f}")
print(f"Ersatzwert für fehlende city: {city_fill_value}")
print("Transformierte Daten, gerundet:")
print(transformed_data.round(1).iloc[:5, :5])



### **1.3. Kategorien kodieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_01_03.jpg?v=1784796130" width="250">



>* Kategorien müssen numerisch kodiert werden
>* Nominal und ordinal passend unterscheiden

>* One-Hot kodiert nominale Kategorien ohne Rangfolge
>* Viele Kategorien erzeugen breite Datensätze

>* Ordinale Kategorien nur bei klarer Reihenfolge kodieren
>* Unbekannte Kategorien robust und fachlich behandeln



In [ ]:
#@title Python-Code - Kategorien kodieren

# Dieses Beispiel kodiert kategoriale Merkmale.
# One-Hot-Kodierung vermeidet künstliche Reihenfolgen.
# Die Ausgabe zeigt neue numerische Spalten.

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Kleine Beispieldaten zeigen nominale Kategorien ohne Reihenfolge.
data = pd.DataFrame(
    {
        "payment": ["Karte", "PayPal", "Rechnung", "Karte"],
        "contract": ["Monat", "Jahr", "Monat", "Jahr"],
    }
)

# Der Encoder ignoriert später unbekannte Kategorien kontrolliert.
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
preprocessor = ColumnTransformer(
    [("category_encoder", encoder, ["payment", "contract"])]
)

# Fit lernt die Kategorien, transform wandelt sie numerisch um.
encoded_array = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()
encoded_data = pd.DataFrame(encoded_array, columns=feature_names)

# Eine einfache Prüfung macht die erwartete Form sichtbar.
expected_rows = len(data)
if encoded_data.shape[0] != expected_rows:
    raise ValueError("Die Anzahl der Zeilen stimmt nicht.")

print("Originale kategoriale Spalten:")
print(data.head(4).to_string(index=False))
print("Kodierte numerische Spalten:")
print(encoded_data.head(4).astype(int).to_string(index=False))
print("Merkmalsnamen nach One-Hot-Kodierung:")
print(", ".join(feature_names))



## **2. Spalten gezielt verbinden**

### **2.1. Seltene Kategorien bündeln**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_02_01.jpg?v=1784796132" width="250">



>* Seltene Kategorien liefern oft instabile Muster
>* Bündeln reduziert Merkmale und bewahrt Information

>* ColumnTransformer bündelt seltene Kategorien gezielt
>* Weniger Sparse-Spalten, stabilere Modellmuster

>* Seltenheit nur aus Trainingsdaten lernen
>* Pipeline verhindert Leckage und bleibt nachvollziehbar



In [ ]:
#@title Python-Code - Seltene Kategorien bündeln

# Dieses Beispiel bündelt seltene Kategorien automatisch.
# ColumnTransformer verbindet kategoriale und numerische Schritte.
# Die Ausgabe zeigt stabile Merkmalsnamen.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Kleine Trainingsdaten zeigen häufige und seltene Städte.
data = pd.DataFrame(
    {
        "city": ["Berlin", "Berlin", "Hamburg", "Hamburg", "Ulm", "Kiel"],
        "age": [34, 41, 29, 52, 46, 38],
        "spend_eur": [120, 180, 90, 210, 75, 60],
    }
)

# Diese Spalten werden unterschiedlich vorbereitet.
numeric_features = ["age", "spend_eur"]
categorical_features = ["city"]

# Seltene Kategorien werden in eine gemeinsame Gruppe gelegt.
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(min_frequency=2, sparse_output=False)),
    ]
)

# Numerische Werte werden imputiert und skaliert.
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# ColumnTransformer verbindet beide Vorbereitungen spaltenweise.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# Die Häufigkeiten werden nur aus diesen Trainingsdaten gelernt.
transformed = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()

# Neue Daten nutzen dieselbe gelernte Bündelungslogik.
new_data = pd.DataFrame(
    {"city": ["Berlin", "Kiel"], "age": [30, 44], "spend_eur": [110, 70]}
)
new_transformed = preprocessor.transform(new_data)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Ursprüngliche Spalten: {list(data.columns)}")
print(f"Neue Merkmale: {list(feature_names)}")
print(f"Trainingsform nach Transformation: {transformed.shape}")
print(f"Neue Daten nach Transformation: {new_transformed.shape}")
print("Kiel landet in der gelernten Sammelspalte für seltene Städte.")



### **2.2. Spalten gezielt transformieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_02_02.jpg?v=1784796134" width="250">



>* Numerische und kategoriale Spalten getrennt vorbereiten
>* Passende Transformationen bewusst pro Spalte wählen

>* ColumnTransformer ordnet Spalten passende Verarbeitung zu
>* Datentypen behalten ihre geeigneten Transformationen

>* Gezielte Transformation verhindert typische Vorverarbeitungsfehler
>* ColumnTransformer schafft konsistente, robuste Pipelines



In [ ]:
#@title Python-Code - Spalten gezielt transformieren

# Dieses Beispiel transformiert Spalten gezielt.
# Numerische und kategoriale Merkmale erhalten passende Schritte.
# Am Ende sehen wir neue Merkmalsnamen.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Ein kleiner Datensatz enthält gemischte Spaltentypen.
data = pd.DataFrame(
    {
        "age": [25, 40, None, 35],
        "income_eur": [32000, 58000, 45000, None],
        "region": ["Nord", "Süd", "Nord", None],
        "contract": ["Basis", "Premium", "Basis", "Premium"],
    }
)

# Diese Listen steuern, welche Spalten wohin gehen.
numeric_features = ["age", "income_eur"]
categorical_features = ["region", "contract"]

# Numerische Spalten werden ergänzt und anschließend skaliert.
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Kategoriale Spalten werden ergänzt und anschließend kodiert.
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(sparse_output=False)),
    ]
)

# Der ColumnTransformer verbindet Spaltenlisten mit passenden Pipelines.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# Fit lernt nur die nötigen Werte aus den Daten.
transformed_array = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()

# Eine kleine Tabelle macht das Ergebnis gut lesbar.
transformed_data = pd.DataFrame(
    transformed_array,
    columns=feature_names,
)

# Diese Prüfung zeigt, dass alle Zeilen erhalten bleiben.
if transformed_data.shape[0] != data.shape[0]:
    raise ValueError("Die Anzahl der Zeilen passt nicht.")

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Vorher: {data.shape[1]} Spalten, nachher: {transformed_data.shape[1]} Spalten")
print("Neue Merkmale:", list(feature_names))
print(transformed_data.round(2).head(4).to_string(index=False))



### **2.3. Pipeline verbinden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_02_03.jpg?v=1784796136" width="250">



>* Pipeline verbindet Vorverarbeitung und Modell zuverlässig
>* Transformationen laufen konsistent in richtiger Reihenfolge

>* ColumnTransformer verarbeitet Spalten passend vor dem Modell
>* Pipeline verhindert Datenleckage durch Trainingswerte

>* Pipelines machen ML-Abläufe reproduzierbar und wartbar
>* Gleiche Vorverarbeitung für Vergleich und Produktion



In [ ]:
#@title Python-Code - Pipeline verbinden

# Diese Pipeline verbindet Vorverarbeitung und Modell.
# ColumnTransformer behandelt Spalten passend nach Typ.
# Die Ausgabe zeigt Genauigkeit und Merkmalsnamen.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Rohdaten enthalten numerische und kategoriale Spalten.
data = pd.DataFrame(
    {
        "area_m2": [45, 80, 60, 120, 35, 95, 70, 110, 50, 85, 40, 100],
        "year_built": [1990, 2010, 2005, 2018, 1975, 2012, 2000, 2020, 1985, 2015, 1980, 2019],
        "district": ["Nord", "Sued", "Nord", "West", "Ost", "Sued", "West", "West", "Nord", "Sued", "Ost", "West"],
        "building_type": ["Altbau", "Neubau", "Altbau", "Neubau", "Altbau", "Neubau", "Altbau", "Neubau", "Altbau", "Neubau", "Altbau", "Neubau"],
    }
)

target = pd.Series([0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], name="expensive")

# Diese Prüfung macht die Beispielannahme sichtbar.
if len(data) != len(target):
    raise ValueError("Daten und Zielwerte müssen gleich lang sein.")

numeric_features = ["area_m2", "year_built"]
categorical_features = ["district", "building_type"]

# Jeder Spaltentyp bekommt seine eigene kleine Pipeline.
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer()), ("scaler", StandardScaler())]
)

categorical_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

# ColumnTransformer führt beide Zweige wieder zusammen.
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)]
)

# Die Gesamtpipeline verbindet Vorverarbeitung und Modell.
model_pipeline = Pipeline(
    steps=[("preprocess", preprocessor), ("model", LogisticRegression(random_state=42))]
)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.25, random_state=42, stratify=target
)

model_pipeline.fit(X_train, y_train)
predictions = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

feature_names = model_pipeline.named_steps["preprocess"].get_feature_names_out()
shown_features = ", ".join(feature_names[:5])

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit der vollständigen Pipeline: {accuracy:.2f}")
print(f"Erste erzeugte Merkmalsnamen: {shown_features}")
print("Pipeline-Schritte: preprocess -> model")



## **3. Pipeline prüfen**

### **3.1. Schritte gezielt prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_03_01.jpg?v=1784796138" width="250">



>* Pipeline-Schritte einzeln sichtbar machen
>* Fehlerursachen früher und gezielter finden

>* Geplante Schritte von gelernten Zuständen unterscheiden
>* Gelernte Werte fachlich und transparent prüfen

>* Problemstelle im Pipeline-Ablauf gezielt finden
>* Datenfluss prüfen und Modelle erklärbarer machen



In [ ]:
#@title Python-Code - Schritte gezielt prüfen

# Wir prüfen gezielt gelernte Pipeline-Schritte.
# ColumnTransformer trennt numerische und kategoriale Verarbeitung.
# Die Ausgabe zeigt Parameter und Merkmalsnamen.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Ein kleiner Datensatz macht jeden Schritt gut nachvollziehbar.
housing = pd.DataFrame(
    {
        "area_m2": [55, 80, np.nan, 120, 65, 95],
        "year_built": [1990, 2005, 1980, np.nan, 2010, 1975],
        "district": ["Mitte", "Nord", "Sued", "Mitte", None, "Nord"],
        "energy_class": ["B", "A", "C", "B", "A", None],
    }
)

# Das Ziel ist hier nur ein kleines Klassifikationsbeispiel.
target = np.array([0, 1, 0, 1, 1, 0])

# Diese Spalten werden später getrennt verarbeitet.
numeric_features = ["area_m2", "year_built"]
categorical_features = ["district", "energy_class"]

# Numerische Werte werden ersetzt und danach skaliert.
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Kategoriale Werte werden ersetzt und danach codiert.
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Der ColumnTransformer verbindet beide Spaltenwege.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# Die vollständige Pipeline endet mit einem einfachen Modell.
model_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(random_state=42, max_iter=200)),
    ]
)

# Nach fit enthalten die Schritte gelernte Informationen.
model_pipeline.fit(housing, target)

# Eine einfache Prüfung schützt vor unerwarteten Spaltenzahlen.
feature_names = model_pipeline.named_steps["preprocess"].get_feature_names_out()
transformed = model_pipeline.named_steps["preprocess"].transform(housing)

if transformed.shape[1] != len(feature_names):
    raise ValueError("Die Anzahl der Merkmalsnamen passt nicht.")

# Jetzt greifen wir gezielt auf verschachtelte Schritte zu.
num_imputer = model_pipeline.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"]
cat_encoder = model_pipeline.named_steps["preprocess"].named_transformers_["cat"].named_steps["encoder"]

# Die Ausgabe bleibt kurz und zeigt die wichtigsten Prüfstellen.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Pipeline-Schritte: {list(model_pipeline.named_steps.keys())}")
print(f"Numerische Ersatzwerte: {np.round(num_imputer.statistics_, 1).tolist()}")
print(f"Kategorien district: {cat_encoder.categories_[0].tolist()}")
print(f"Transformierte Form: {transformed.shape[0]} Zeilen, {transformed.shape[1]} Spalten")
print(f"Erste Merkmalsnamen: {feature_names[:5].tolist()}")



### **3.2. Merkmalsnamen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_03_02.jpg?v=1784796140" width="250">



>* Transformationen erzeugen neue, benannte Modellmerkmale
>* Merkmalsnamen machen Modellverhalten interpretierbar

>* Transformierte Merkmale haben neue Reihenfolgen und Namen
>* Encoder und Imputer erzeugen fachlich wichtige Signale

>* Merkmalsnamen als Qualitätskontrolle nutzen
>* Interpretation, Fairness und Kommunikation verbessern



In [ ]:
#@title Python-Code - Merkmalsnamen verstehen

# Dieses Beispiel zeigt transformierte Merkmalsnamen.
# ColumnTransformer verbindet numerische und kategoriale Schritte.
# Die Ausgabe macht neue Modellspalten sichtbar.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Kleine Beispieldaten enthalten Zahlen, Kategorien und fehlende Werte.
data = pd.DataFrame(
    {
        "age": [25, 40, None, 35],
        "income": [30000, 52000, 41000, None],
        "contract": ["monthly", "yearly", "monthly", "trial"],
        "city": ["Berlin", "Hamburg", "Berlin", "Munich"],
    }
)

numeric_features = ["age", "income"]
categorical_features = ["contract", "city"]

# Numerische Spalten werden imputiert und anschließend skaliert.
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Kategoriale Spalten werden imputiert und in Dummyspalten zerlegt.
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(sparse_output=False)),
    ]
)

# Der ColumnTransformer führt beide Teilpipelines zusammen.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

transformed_data = preprocessor.fit_transform(data)
feature_names = preprocessor.get_feature_names_out()

if transformed_data.shape[1] != len(feature_names):
    raise ValueError("Jede Ausgabespalte braucht genau einen Namen.")

preview = pd.DataFrame(
    transformed_data,
    columns=feature_names,
).round(2)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Ursprüngliche Spalten: {list(data.columns)}")
print(f"Transformierte Merkmale: {list(feature_names)}")
print("Erste zwei transformierte Zeilen:")
print(preview.head(2).iloc[:, :5])



### **3.3. Ende zu Ende**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_09/Lecture_B/image_03_03.jpg?v=1784796141" width="250">



>* Datenweg bis zur Vorhersage nachvollziehen
>* Fehlzuordnungen früh erkennen und Bedeutung sichern

>* Datenfluss durch alle Pipeline-Schritte prüfen
>* Merkmalsnamen mit Rohdaten verknüpfen

>* Pipeline-Schritte und Merkmalsnamen nachvollziehen
>* Mehr Vertrauen, Wartbarkeit und Transparenz gewinnen



In [ ]:
#@title Python-Code - Ende zu Ende

# Diese Prüfung verfolgt eine Pipeline vollständig.
# Verschachtelte Schritte und Merkmalsnamen werden sichtbar.
# Die Ausgabe zeigt Modellgüte und transformierte Spalten.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Ein kleiner Kundendatensatz enthält Zahlen, Kategorien und Lücken.
customer_data = pd.DataFrame(
    {
        "age": [22, 35, np.nan, 52, 46, 28, 41, 60, 33, 48, 25, 55],
        "income": [32, 58, 45, 88, np.nan, 40, 72, 95, 50, 80, 36, 90],
        "contract": ["basic", "plus", "basic", "premium", "plus", "basic", "premium", "premium", "plus", "premium", "basic", "plus"],
        "region": ["north", "south", "north", "west", "south", "east", "west", "north", "east", "west", "south", "north"],
    }
)

# Das Ziel beschreibt, ob ein Kunde verlängert hat.
target = np.array([0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1])

# Eine einfache Prüfung schützt vor unpassenden Zeilenzahlen.
if len(customer_data) != len(target):
    raise ValueError("Daten und Ziel haben unterschiedlich viele Zeilen.")

# Die Spaltenlisten steuern den ColumnTransformer eindeutig.
numeric_features = ["age", "income"]
categorical_features = ["contract", "region"]

# Numerische Werte werden imputiert und anschließend skaliert.
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer()), ("scaler", StandardScaler())]
)

# Kategorien werden imputiert und danach als Dummy-Spalten codiert.
categorical_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

# Der ColumnTransformer verbindet beide Spaltenwege.
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, numeric_features), ("cat", categorical_pipeline, categorical_features)]
)

# Die vollständige Pipeline endet mit einem kleinen Klassifikationsmodell.
model_pipeline = Pipeline(
    steps=[("preprocess", preprocessor), ("model", LogisticRegression(max_iter=200, random_state=42))]
)

# Der Split hält die Modellprüfung von den Trainingsdaten getrennt.
X_train, X_test, y_train, y_test = train_test_split(
    customer_data, target, test_size=0.33, random_state=42, stratify=target
)

# Fit trainiert Imputer, Encoder, Skalierer und Modell gemeinsam.
model_pipeline.fit(X_train, y_train)

# Die transformierten Namen verbinden Rohdaten mit Modellinput.
feature_names = model_pipeline.named_steps["preprocess"].get_feature_names_out()

# Verschachtelte Parameter zeigen, welcher Imputer verwendet wird.
imputer_strategy = model_pipeline.get_params()["preprocess__num__imputer__strategy"]

# Die transformierte Matrix zeigt die endgültige Modell-Eingabeform.
transformed_train = model_pipeline.named_steps["preprocess"].transform(X_train)

# Die Ausgabe bleibt kurz und prüft die Pipeline Ende zu Ende.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Pipeline-Schritte: {list(model_pipeline.named_steps.keys())}")
print(f"Numerische Imputer-Strategie: {imputer_strategy}")
print(f"Transformierte Trainingsform: {transformed_train.shape}")
print(f"Erste Merkmalsnamen: {list(feature_names[:5])}")
print(f"Testgenauigkeit: {model_pipeline.score(X_test, y_test):.2f}")



# <font color="#418FDE" size="6.5" uppercase>**Pipelines bauen**</font>


In this lecture, you learned to:
- Wenden Skalierer, Imputer und Encoder passend auf numerische und kategoriale Spalten an. 
- Verbinden Spaltentransformationen mit ColumnTransformer und Pipeline. 
- Untersuchen Pipeline-Schritte, verschachtelte Parameter und Merkmalsnamen. 

In the next Module (Module 10), we will go over 'Klassische Modelle'